In [146]:
import os
import re
import time
import json
import requests
import zipfile
import io
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
)

In [147]:
# =========================
# LOAD API KEYS
# =========================
load_dotenv()
BLS_API_KEY = os.getenv("BLS_API_KEY")
TARGET_SERIES = "CUUR0000SA0"   # BLS NSA headline CPI-U all items
START_DATE = "2011-12-01"       # include Dec 2011 so Jan 2012 can be chained
END_DATE = None                # None = latest available
DROP_MISSING_INDEX_ROWS = True

BLS_URL = "https://api.bls.gov/publicAPI/v2/timeseries/data/"


def fetch_bls_one_series_wide(series_id, start_year, end_year, bls_api_key=None):
    df = fetch_bls_series([series_id], start_year, end_year, bls_api_key=bls_api_key)
    if df.empty:
        raise ValueError(f"No observations returned for {series_id}")
    out = df[["date", "value"]].rename(columns={"value": series_id}).sort_values("date")
    return out

def fetch_bls_series(series_ids, start_year, end_year, bls_api_key=None):
    payload = {
        "seriesid": series_ids,
        "startyear": str(start_year),
        "endyear": str(end_year),
    }
    if bls_api_key:
        payload["registrationkey"] = bls_api_key

    r = requests.post(
        BLS_URL,
        data=json.dumps(payload),
        headers={"Content-type": "application/json"},
        timeout=60
    )
    r.raise_for_status()
    data = r.json()

    status = data.get("status")
    if status != "REQUEST_SUCCEEDED":
        raise ValueError(data)

    rows = []
    returned_series = data.get("Results", {}).get("series", [])

    for s in returned_series:
        sid = s.get("seriesID")
        observations = s.get("data", [])

        for obs in observations:
            period = obs.get("period", "")
            if period.startswith("M") and period != "M13":
                rows.append({
                    "series_id": sid,
                    "date": pd.Timestamp(
                        year=int(obs["year"]),
                        month=int(period[1:]),
                        day=1
                    ),
                    "value": pd.to_numeric(obs.get("value"), errors="coerce")
                })

    return pd.DataFrame(rows)

# =========================
# DOWNLOAD TARGET CPI
# =========================
start_year = pd.Timestamp(START_DATE).year
end_year = pd.Timestamp.today().year if END_DATE is None else pd.Timestamp(END_DATE).year

target = fetch_bls_one_series_wide(
    TARGET_SERIES,
    start_year=start_year,
    end_year=end_year,
    bls_api_key=BLS_API_KEY
)
target = target.rename(columns={TARGET_SERIES: "official_cpi"})
target = target[target["date"] >= pd.Timestamp(START_DATE)].copy()
if END_DATE is not None:
    target = target[target["date"] <= pd.Timestamp(END_DATE)].copy()
#target = target.dropna()
target = target.set_index('date')

In [148]:
target.tail(12)

,official_cpi
date,
2025-10-01,NaN
2025-11-01,324.122
2025-12-01,324.054
2026-01-01,325.252
2026-02-01,326.785
2026-03-01,330.213
2026-04-01,333.020
2026-05-01,335.123
2026-06-01,333.952


In [149]:
target.index = pd.to_datetime(target.index)
#target.index = target.index.strftime("%Y-%m")
target = target.asfreq("MS")
target.tail(12)

,official_cpi
date,
2025-10-01,NaN
2025-11-01,324.122
2025-12-01,324.054
2026-01-01,325.252
2026-02-01,326.785
2026-03-01,330.213
2026-04-01,333.020
2026-05-01,335.123
2026-06-01,333.952


In [150]:
## Getting WTI Crude prices from FRED

FRED_API_KEY = os.getenv("FRED_API_KEY")

url = "https://api.stlouisfed.org/fred/series/observations"

params = {
    "series_id": "MCOILWTICO",
    "api_key": FRED_API_KEY,
    "file_type": "json",
    "observation_start": "2000-01-01"
}

response = requests.get(url, params=params)
data = response.json()

oil = pd.DataFrame(data["observations"])[["date", "value"]]
oil["date"] = pd.to_datetime(oil["date"])
oil["value"] = pd.to_numeric(oil["value"], errors="coerce")

oil = oil.rename(columns={"value": "oil_price"})
oil = oil.set_index("date")

In [151]:
oil.tail(12)

,oil_price
date,
2025-10-01,60.89
2025-11-01,60.06
2025-12-01,57.97
2026-01-01,60.04
2026-02-01,64.51
2026-03-01,91.38
2026-04-01,100.32
2026-05-01,102.13
2026-06-01,84.81


In [152]:
oil.index = oil.index.to_period("M").to_timestamp()
oil['oil_previous_month'] = oil['oil_price'].shift(1)
oil['oil_MoM'] = oil['oil_price'].pct_change(1)*100
oil['oil_MoM_1m_lag'] = oil['oil_MoM'].shift(1)
oil['oil_MoM_2m_lag'] = oil['oil_MoM'].shift(2)
oil['oil_MoM_3m_lag'] = oil['oil_MoM'].shift(3)
oil = oil.dropna()
oil.tail(12)

,oil_price,oil_previous_month,oil_MoM,oil_MoM_1m_lag,oil_MoM_2m_lag,oil_MoM_3m_lag
date,,,,,,
2025-10-01,60.89,63.96,-4.799875,-1.387604,-5.161573,0.322723
2025-11-01,60.06,60.89,-1.363114,-4.799875,-1.387604,-5.161573
2025-12-01,57.97,60.06,-3.479853,-1.363114,-4.799875,-1.387604
2026-01-01,60.04,57.97,3.570812,-3.479853,-1.363114,-4.799875
2026-02-01,64.51,60.04,7.445037,3.570812,-3.479853,-1.363114
2026-03-01,91.38,64.51,41.652457,7.445037,3.570812,-3.479853
2026-04-01,100.32,91.38,9.783322,41.652457,7.445037,3.570812
2026-05-01,102.13,100.32,1.804226,9.783322,41.652457,7.445037
2026-06-01,84.81,102.13,-16.958778,1.804226,9.783322,41.652457


In [153]:
# ##Retrieving UNRATE data

# unrate_nsa = pd.read_csv(
#     "https://fred.stlouisfed.org/graph/fredgraph.csv?id=UNRATENSA",
#     parse_dates=["observation_date"],
#     index_col="observation_date",
# )

# unrate_nsa = unrate_nsa.rename(
#     columns={"UNRATENSA": "UNRATE"}
# )
# unrate_nsa.loc[pd.Timestamp("2025-10-01"), "UNRATE"] = 4.3
# unrate_nsa['UNRATE_lag1'] = unrate_nsa['UNRATE'].shift(1)
# unrate_nsa['UNRATE_lag2'] = unrate_nsa['UNRATE'].shift(2)
# unrate_nsa['UNRATE_lag3'] = unrate_nsa['UNRATE'].shift(3)
# unrate_nsa = unrate_nsa['2011-12-01':]
# unrate_nsa.tail(13)

In [154]:
##Retrieving NFP data

df_housing = pd.read_csv(
    "https://fred.stlouisfed.org/graph/fredgraph.csv?id=HPIPONM226N",
    parse_dates=["observation_date"],
    index_col="observation_date",
)

df_housing = df_housing.rename(
    columns={"HPIPONM226N": "Housing"}
)

df_housing['Housing_lag3'] = df_housing['Housing'].shift(3)
df_housing['Housing_lag6'] = df_housing['Housing'].shift(6)
df_housing['Housing_lag9'] = df_housing['Housing'].shift(9)
df_housing['Housing_lag12'] = df_housing['Housing'].shift(12)

df_housing = df_housing['2011-12-01':]
df_housing

,Housing,Housing_lag3,Housing_lag6,Housing_lag9,Housing_lag12
observation_date,,,,,
2011-12-01,177.60,180.30,180.95,176.19,180.78
2012-01-01,175.80,178.70,181.29,178.16,178.48
2012-02-01,176.49,178.67,180.18,179.37,177.11
2012-03-01,179.10,177.60,180.30,180.95,176.19
2012-04-01,181.59,175.80,178.70,181.29,178.16
...,...,...,...,...,...
2026-03-01,445.42,437.24,438.23,441.90,436.56
2026-04-01,448.18,437.23,438.04,440.33,439.09
2026-05-01,452.42,441.02,438.55,439.84,441.89


In [155]:
df_housing.isna().sum()

Housing          0
Housing_lag3     0
Housing_lag6     0
Housing_lag9     0
Housing_lag12    0
dtype: int64

In [156]:
df_cpi_zori_oil = pd.concat([target, oil, df_housing], axis=1, sort=True)
df_cpi_zori_oil = df_cpi_zori_oil.asfreq("MS")
df_cpi_zori_oil['last_month_cpi'] = df_cpi_zori_oil['official_cpi'].shift(1)
df_cpi_zori_oil = df_cpi_zori_oil['2011-12-01':]
#df_cpi_zori_oil = df_cpi_zori_oil.dropna()

In [157]:
df_cpi_zori_oil.head(13)

,official_cpi,oil_price,oil_previous_month,oil_MoM,oil_MoM_1m_lag,oil_MoM_2m_lag,oil_MoM_3m_lag,Housing,Housing_lag3,Housing_lag6,Housing_lag9,Housing_lag12,last_month_cpi
2011-12-01,225.672,98.56,97.16,1.440922,12.557924,0.935454,-0.938260,177.60,180.30,180.95,176.19,180.78,NaN
2012-01-01,226.665,100.27,98.56,1.734984,1.440922,12.557924,0.935454,175.80,178.70,181.29,178.16,178.48,225.672
2012-02-01,227.663,102.20,100.27,1.924803,1.734984,1.440922,12.557924,176.49,178.67,180.18,179.37,177.11,226.665
2012-03-01,229.392,106.16,102.20,3.874755,1.924803,1.734984,1.440922,179.10,177.60,180.30,180.95,176.19,227.663
2012-04-01,230.085,103.32,106.16,-2.675207,3.874755,1.924803,1.734984,181.59,175.80,178.70,181.29,178.16,229.392
2012-05-01,229.815,94.66,103.32,-8.381727,-2.675207,3.874755,1.924803,184.53,176.49,178.67,180.18,179.37,230.085
2012-06-01,229.478,82.30,94.66,-13.057258,-8.381727,-2.675207,3.874755,186.46,179.10,177.60,180.30,180.95,229.815
2012-07-01,229.104,87.90,82.30,6.804374,-13.057258,-8.381727,-2.675207,186.79,181.59,175.80,178.70,181.29,229.478
2012-08-01,230.379,94.13,87.90,7.087600,6.804374,-13.057258,-8.381727,187.22,184.53,176.49,178.67,180.18,229.104
2012-09-01,231.407,94.51,94.13,0.403697,7.087600,6.804374,-13.057258,186.92,186.46,179.10,177.60,180.30,230.379


In [158]:
df_cpi_zori_oil['month'] = df_cpi_zori_oil.index.month
df_cpi_zori_oil['month'] = df_cpi_zori_oil['month'].astype(int)

##Fix to fill last 2 months for housing with lags
df_cpi_zori_oil.loc[df_cpi_zori_oil.index[-2:], 'Housing_lag3'] = (df_cpi_zori_oil['Housing'].iloc[[-5, -4]].values)
df_cpi_zori_oil.loc[df_cpi_zori_oil.index[-2:], 'Housing_lag6'] = (df_cpi_zori_oil['Housing'].iloc[[-8, -7]].values)
df_cpi_zori_oil.loc[df_cpi_zori_oil.index[-2:], 'Housing_lag9'] = (df_cpi_zori_oil['Housing'].iloc[[-11, -10]].values)
#df_cpi_zori_oil['Housing_lag6'] = df_cpi_zori_oil['Housing'].shift(6)
#df_cpi_zori_oil['Housing_lag9'] = df_cpi_zori_oil['Housing'].shift(9)

df_cpi_zori_oil.tail(12)

,official_cpi,oil_price,oil_previous_month,oil_MoM,oil_MoM_1m_lag,oil_MoM_2m_lag,oil_MoM_3m_lag,Housing,Housing_lag3,Housing_lag6,Housing_lag9,Housing_lag12,last_month_cpi,month
2025-10-01,NaN,60.89,63.96,-4.799875,-1.387604,-5.161573,0.322723,438.04,440.33,439.09,428.68,429.51,324.800,10
2025-11-01,324.122,60.06,60.89,-1.363114,-4.799875,-1.387604,-5.161573,438.55,439.84,441.89,432.52,428.74,NaN,11
2025-12-01,324.054,57.97,60.06,-3.479853,-1.363114,-4.799875,-1.387604,437.24,438.23,441.90,436.56,428.05,324.122,12
2026-01-01,325.252,60.04,57.97,3.570812,-3.479853,-1.363114,-4.799875,437.23,438.04,440.33,439.09,428.68,324.054,1
2026-02-01,326.785,64.51,60.04,7.445037,3.570812,-3.479853,-1.363114,441.02,438.55,439.84,441.89,432.52,325.252,2
2026-03-01,330.213,91.38,64.51,41.652457,7.445037,3.570812,-3.479853,445.42,437.24,438.23,441.90,436.56,326.785,3
2026-04-01,333.020,100.32,91.38,9.783322,41.652457,7.445037,3.570812,448.18,437.23,438.04,440.33,439.09,330.213,4
2026-05-01,335.123,102.13,100.32,1.804226,9.783322,41.652457,7.445037,452.42,441.02,438.55,439.84,441.89,333.020,5
2026-06-01,333.952,84.81,102.13,-16.958778,1.804226,9.783322,41.652457,452.20,445.42,437.24,438.23,441.90,335.123,6
2026-07-01,333.918,80.46,84.81,-5.129112,-16.958778,1.804226,9.783322,451.60,448.18,437.23,438.04,440.33,333.952,7


In [159]:
##Testing SARIMAX with multiple exogenous variables

target = "official_cpi"
features_cols = [
    #"oil_price",
    "oil_previous_month",
    #"oil_MoM_1m_lag",
    #"oil_MoM_2m_lag",
    #"oil_MoM_3m_lag",
    #"Housing_lag3",
    #"Housing_lag6",
    #"Housing_lag9",
    #"last_month_cpi",
    #"month"
]

In [160]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Work on a copy so the original dataframe is not changed.
#sarimax_data = df_cpi_zori_oil['2011-12-01':'2025-09-01'].copy()
sarimax_data = df_cpi_zori_oil.copy()

# Ensure the index is a DatetimeIndex.
sarimax_data.index = pd.to_datetime(
    sarimax_data.index
)

# Define the endogenous target and exogenous variables.
endog = sarimax_data[target]
exog = sarimax_data[features_cols]

# Fit seasonal SARIMAX on the complete monthly time axis.
sarimax_model = SARIMAX(
    endog=endog,
    exog=exog,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    trend="c",
    missing="none",
)


# Estimate the model parameters.
sarimax_results = sarimax_model.fit(
    method="lbfgs",
    maxiter=500,
    disp=False,
)

# # Display the fitted model summary.
# print(sarimax_results.summary())

sarimax_prediction_result = sarimax_results.get_prediction(
    start=13,
    end=len(sarimax_data) - 1,
    exog=exog[13:],
)


# Extract the predicted values.
sarimax_predictions = sarimax_prediction_result.predicted_mean


# Extract 95% confidence intervals.
sarimax_conf_int = sarimax_prediction_result.conf_int()


# Force the prediction index to match the original data index.
sarimax_predictions.index = sarimax_data.index[13:]
sarimax_conf_int.index = sarimax_data.index[13:]


# Create a dataframe containing official values, predictions, and errors.
sarimax_results_df = pd.DataFrame({
    "official_cpi": endog,
    "sarimax_prediction": sarimax_predictions
})


# Calculate the signed prediction error.
# Positive values mean the model underpredicted official_cpi.
sarimax_results_df["error"] = (
    sarimax_results_df["official_cpi"]
    - sarimax_results_df["sarimax_prediction"]
)


# Calculate the absolute error.
sarimax_results_df["absolute_error"] = (
    sarimax_results_df["error"].abs()
)


# Keep only rows where both the official value and prediction exist.
# This excludes the row with missing official_cpi.
valid_results = sarimax_results_df.apply(pd.to_numeric,errors='coerce').dropna(subset=["official_cpi", "sarimax_prediction"])
# valid_results = sarimax_results_df.dropna(
#     subset=["official_cpi", "sarimax_prediction"]
# )


# Calculate MAE.
mae = mean_absolute_error(
    valid_results["official_cpi"],
    valid_results["sarimax_prediction"],
)


# Calculate RMSE.
rmse = root_mean_squared_error(
        valid_results["official_cpi"],
        valid_results["sarimax_prediction"],
    )



# Print the model summary and metrics.
print(f"\nMAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")


# Display the dataframe.
results = sarimax_results_df.copy()
results = results.dropna()
results


MAE:  0.50
RMSE: 0.67


,official_cpi,sarimax_prediction,error,absolute_error
2013-01-01,230.280,230.650070,-0.370070,0.370070
2013-02-01,232.166,231.584928,0.581072,0.581072
2013-03-01,232.773,233.902824,-1.129824,1.129824
2013-04-01,232.531,233.067861,-0.536861,0.536861
2013-05-01,232.945,232.287109,0.657891,0.657891
...,...,...,...,...
2026-05-01,335.123,334.851428,0.271572,0.271572
2026-06-01,333.952,336.627864,-2.675864,2.675864
2026-07-01,333.918,333.344954,0.573046,0.573046
2026-08-01,334.980,334.355835,0.624165,0.624165


In [161]:
next_forecast = sarimax_results.forecast(
    steps=1,
    exog=df_cpi_zori_oil[['oil_price']].iloc[[-1]]
    #exog = df_cpi_zori_oil[['oil_MoM_1m_lag','NFP', 'NFP_lag1']].iloc[[-1]]
#     exog = [
#     df_cpi_zori_oil['oil_MoM_1m_lag'].iloc[-1],
#     df_cpi_zori_oil['Housing'].iloc[-5],
#     df_cpi_zori_oil['Housing'].iloc[-8]
# ]
)

In [162]:
#last_month = df_cpi_zori_oil.index[-1].month
#next_month = last_month + pd.DateOffset(months=1)
next_month = (pd.to_datetime(df_cpi_zori_oil.index[-1]) + pd.DateOffset(months=1)).strftime("%B")
next_cpi = round(next_forecast.iloc[0],1)
yoy = round((next_cpi / df_cpi_zori_oil['official_cpi'].iloc[-12]),1)
print(next_month + ' US CPI print: ' + str(next_cpi))
print('YoY Inflation: ' + str(yoy) + '%')

October US CPI print: 338.4
YoY Inflation: nan%


In [163]:
import pymars as earth

In [164]:
##Trying MARS ML model

df_cpi_zori_oil['CPI_lag12'] = df_cpi_zori_oil['official_cpi'].shift(12)

features_cols = [
    "oil_previous_month",
    #"oil_price",
    "CPI_lag12",
    "last_month_cpi",
    "Housing_lag6",
    "Housing_lag9",
    #"Housing_lag12",
    "month",
]

mars_data = df_cpi_zori_oil[
    ["official_cpi"] + features_cols
].dropna()

X = mars_data[features_cols]
y = mars_data["official_cpi"]

mars_model = earth.Earth(
    max_degree=1,
    penalty=3.0,
)

mars_model.fit(X, y)

mars_predictions = mars_model.predict(X)

mars_results = pd.DataFrame(
    {
        "official_cpi": y,
        "mars_prediction": mars_predictions,
    },
    index=y.index,
)

mars_results["error"] = (
    mars_results["official_cpi"]
    - mars_results["mars_prediction"]
)

mars_results["absolute_error"] = (
    mars_results["error"].abs()
)

# Calculate MAE.
mae = mean_absolute_error(
    mars_results["official_cpi"],
    mars_results["mars_prediction"],
)


# Calculate RMSE.
rmse = root_mean_squared_error(
        mars_results["official_cpi"],
        mars_results["mars_prediction"],
    )


print(f"\nMAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
mars_results


MAE:  0.41
RMSE: 0.53


,official_cpi,mars_prediction,error,absolute_error
2012-12-01,229.601,229.075047,0.525953,0.525953
2013-01-01,230.280,229.914547,0.365453,0.365453
2013-02-01,232.166,230.908410,1.257590,1.257590
2013-03-01,232.773,232.325930,0.447070,0.447070
2013-04-01,232.531,232.816561,-0.285561,0.285561
...,...,...,...,...
2026-05-01,335.123,334.777372,0.345628,0.345628
2026-06-01,333.952,334.208631,-0.256631,0.256631
2026-07-01,333.918,333.511520,0.406480,0.406480
2026-08-01,334.980,335.258907,-0.278907,0.278907


In [165]:
last_date = df_cpi_zori_oil.index.max()
next_date = last_date + pd.offsets.MonthBegin(1)

next_features = pd.DataFrame(
    {
        "oil_previous_month": [df_cpi_zori_oil["oil_price"].iloc[-1]],
        "CPI_lag12": [df_cpi_zori_oil["official_cpi"].iloc[-11]],
        "last_month_cpi": [df_cpi_zori_oil["official_cpi"].iloc[-1]],
        "Housing_lag6": [df_cpi_zori_oil["Housing"].iloc[-5]],
        "Housing_lag9": [df_cpi_zori_oil["Housing"].iloc[-8]],
        "month": [next_date.month],
    },
    index=[next_date],
)

next_features = next_features[features_cols]
next_forecast = mars_model.predict(next_features)[0]

print(f"Forecast month: {next_date:%Y-%m}")
print(f"MARS CPI forecast: {next_forecast:.4f}")

Forecast month: 2026-10
MARS CPI forecast: 340.2835


In [166]:
ensemble = round((next_cpi+next_forecast)/2,2)
pd.DataFrame({"SARIMAX": next_cpi,
              "MARS": round(next_forecast,2),
              "Ensemble forecast" :ensemble},index=['CPI'])

,SARIMAX,MARS,Ensemble forecast
CPI,338.4,340.28,339.34
